# Hand Gesture Recognition using Convolutional Neural Network (CNN)

## 1. Project Title and Objective
**Project Title:** Hand Gesture Recognition using Convolutional Neural Network (CNN)
**Objective:** Develop a hand gesture recognition model that can accurately identify and classify different hand gestures from image data, enabling intuitive human-computer interaction and gesture-based control systems. This is Task 04 of the SkillCraft Technology Machine Learning Internship.

## 2. Import Required Libraries

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

: 

In [ ]:
DATASET_PATH = "D:\leapGestRecog"

print(f"Dataset path set to: {DATASET_PATH}")

## 4. Explore the Dataset
Let's see how many classes (gestures) we have and count the images in each class.

In [ ]:
if os.path.exists(DATASET_PATH):
    classes = os.listdir(DATASET_PATH)
    print(f"Total classes found: {len(classes)}")
    print("Classes:", classes)
    
    for gesture_class in classes:
        class_path = os.path.join(DATASET_PATH, gesture_class)
        if os.path.isdir(class_path):
            num_images = len(os.listdir(class_path))
            print(f" - {gesture_class}: {num_images} images")
else:
    print("Dataset path not found.")

## 5. Image Preprocessing
- **Resize images:** Standardize the size of all images for the CNN.
- **Normalize pixel values:** Scale pixel values from 0-255 to 0-1.
- **Encode labels:** Convert string labels to numerical format.

In [ ]:
IMG_SIZE = 64 # Resize all images to 64x64

data = []
labels = []

if os.path.exists(DATASET_PATH):
    classes = os.listdir(DATASET_PATH)
    class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}
    idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}
    
    for gesture_class in classes:
        class_path = os.path.join(DATASET_PATH, gesture_class)
        if not os.path.isdir(class_path): continue
            
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            
            # Read image
            img = cv2.imread(img_path)
            if img is None: continue
                
            # Resize image
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            
            data.append(img)
            labels.append(class_to_idx[gesture_class])

# Convert to numpy arrays
data = np.array(data)
labels = np.array(labels)

# Normalize pixel values to [0, 1]
data = data.astype('float32') / 255.0

# Encode labels categorically (One-hot encoding)
if len(labels) > 0:
    labels = to_categorical(labels, num_classes=len(classes))
    print(f"Data shape: {data.shape}")
    print(f"Labels shape: {labels.shape}")

## 6. Visualize Sample Images

In [ ]:
if len(data) > 0:
    plt.figure(figsize=(10, 10))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(cv2.cvtColor(data[i], cv2.COLOR_BGR2RGB))
        # Get the actual class name
        label_idx = np.argmax(labels[i])
        plt.title(idx_to_class[label_idx])
        plt.axis('off')
    plt.show()

## 7. Split Dataset into Training and Testing Sets

In [ ]:
if len(data) > 0:
    X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
    print(f"Training data shape: {X_train.shape}")
    print(f"Testing data shape: {X_test.shape}")

## 8. Build a CNN Model

In [ ]:
if len(data) > 0:
    num_classes = len(classes)

    model = Sequential()
    
    # 1st Convolutional Layer
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # 2nd Convolutional Layer
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # 3rd Convolutional Layer
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # Flatten the results to feed into a DNN
    model.add(Flatten())
    
    # Fully Connected Layer
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5)) # Prevent overfitting
    
    # Output Layer
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()

## 9. Train the Model

In [ ]:
if len(data) > 0:
    EPOCHS = 15
    BATCH_SIZE = 32

    history = model.fit(
        X_train, y_train, 
        epochs=EPOCHS, 
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test)
    )

## 10. Plot Training & Validation Accuracy

In [ ]:
if len(data) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

## 11. Plot Training & Validation Loss

In [ ]:
if len(data) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

## 12. Evaluate the Model
- Accuracy
- Confusion Matrix
- Classification Report

In [ ]:
if len(data) > 0:
    # Predict on test set
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = np.argmax(y_test, axis=1)

    # 1. Accuracy
    test_acc = accuracy_score(y_true, y_pred_classes)
    print(f"Test Accuracy: {test_acc * 100:.2f}%")

    # 2. Confusion Matrix
    cm = confusion_matrix(y_true, y_pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

    # 3. Classification Report
    print("Classification Report:\n")
    print(classification_report(y_true, y_pred_classes, target_names=classes))

## 13. Predict Hand Gestures for New Images
Function to upload and process a new image for prediction.

In [ ]:
def predict_new_image(img_path, model, class_names):
    if not os.path.exists(img_path):
        print(f"Image not found at {img_path}")
        return None, None, None
        
    # Load and preprocess image
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    
    img_normalized = img_resized.astype('float32') / 255.0
    img_expanded = np.expand_dims(img_normalized, axis=0)
    
    # Predict
    prediction = model.predict(img_expanded)
    predicted_class_idx = np.argmax(prediction[0])
    predicted_class_name = class_names[predicted_class_idx]
    confidence = prediction[0][predicted_class_idx]
    
    return img_rgb, predicted_class_name, confidence

## 14. Display the Predicted Image with its Label

In [ ]:
# Example usage:
NEW_IMAGE_PATH = 'path_to_test_image.jpg'

if os.path.exists(NEW_IMAGE_PATH) and len(data) > 0:
    img, label, conf = predict_new_image(NEW_IMAGE_PATH, model, classes)
    
    if img is not None:
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(f"Predicted: {label}\nConfidence: {conf*100:.2f}%")
        plt.axis('off')
        plt.show()
else:
    print(f"Please provide a valid path to an image to test.")

## 15. Conclusion
In this project, we successfully built and trained a Convolutional Neural Network (CNN) to recognize hand gestures from images. We preprocessed the dataset, trained the model, evaluated its performance using accuracy and a confusion matrix, and demonstrated its ability to classify new, unseen images. This completes Task 04 of the SkillCraft Technology Machine Learning Internship.